# 1. API Key 로드

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

# 2. Template 활용하기

지금까지 우리는 메시지 리스트(**메시지 객체**가 담긴 리스트)을 모델에 직접 전달 하였음\
하지만, 보통 이러한 메시지 리스트는 사용자의 입력과 애플리케이션 로직을 조합하여 만들어짐 \
**애플리케이션 로직**은 일반적으로 원시 사용자의 입력을 받아 모델에 전달할 수 있도록 **메시지 리스트**로 변환하는 역할을 함 \
주요 변환 작업에는 system message 추가 또는 사용자 입력(input)을 이용한 **템플릿 포맷팅** 등이 포함 됨

ChatPromptTemplate 클래스는 하나의 템플릿 안에 여러 메시지 role을 지원하며,\
아래의 예제에서 우리는 language변수를 시스템 메시지에 삽입하고, text변수를 사용자 메시지에 삽입 함\
이 프롬프트 템플릿에 대한 입력형태는 딕셔너리이며, 프롬프트 템플릿만 단독으로 사용해보면서 어떤 형태로 모델에 전달되는지 알 수 있음

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0.2
)

## 2.1 Template 활용 전

In [3]:
# question이 먼저 정의됐을 경우
question = "머리가 아파요ㅠㅠ"

user_prompt = f"""\
아래의 사용자 질의에 작성된 통증에 따라 가능성 있는 원인과 해결 방법을 작성해주세요.

사용자 질의: {question}
"""

response = model.invoke(user_prompt)
print(response.content)

머리가 아프다고 하셨죠. 아래는 가능한 원인과 대응 방법을 정리한 일반적인 가이드입니다. 증상의 원인을 확정하려면 의사와 상담이 필요합니다.

1) 가능성 있는 일반적인 원인
- 긴장성 두통: 머리와 목 부근 근육의 긴장으로 양쪽에 뻐근한 통증.
- 편두통: 맥박처럼 아프고, 한쪽이나 특정 부위에 주로 통증. 빛/소리에 예민해짐, 구토 가능.
- 부비동두통: 이마나 눈 주위가 답답하고 압박감. 코막힘이나 콧물 동반.
- 카페인/약물 관련 두통: 카페인을 자주 섭취하다 중단하면 생기는 갈망형 두통 등.
- 탈수나 저혈당, 피로/수면 부족, 스트레스 등 생활 습관 요인.
- 안구 피로/시력 문제: 장시간 화면 보면 악화될 수 있음.
- 드물지만 중요한 원인: 뇌수막염, 뇌출혈, 뇌종양 등의 위험 신호가 동반될 때 나타날 수 있음(아래의 위험 신호 참조).

2) 언제 의사와 상담하거나 응급상황에 가야 하나요? (즉시 병원 방문이 필요할 수 있는 신호)
- 갑작에 극심한 두통이 갑자기 시작되거나 “천둥 치는 듯” 급격하게 심해짐
- 목이 뻣뻣하고 열이 나며 구토가 심한 경우
- 얼굴, 팔 다리에 힘이 없거나 말이 잘 안 되거나 시야가 흐려짐
- 뇌졸중 증상(언어 장애, 한쪽 중심으로 마비, 의식저하), 심한 두통이 이전과 다르게 갑자기 시작
- 최근 머리 부상 후 두통이 심해짐
- 50세 이상 새로 시작된 두통이 이전과 다르게 나타남
- 임신 중이거나 수유 중인 경우, 면역저하 상태인 경우
- 지속적으로 두통이 며칠 이상 지속되거나 자주 재발하는 경우

3) 자가 관리 및 완화 방법 (일반적이고 안전한 방법)
- 수분 섭취와 규칙적인 식사 유지. 카페인도 일정하게 유지하거나 점진적으로 줄이기.
- 충분한 휴식과 수면, 조용하고 어두운 환경에서 쉬기.
- 냉찜질이나 온찜질을 통증 부위에 적용해 보기(두통 유형에 따라 다름).
- 화면 시간 줄이기, 20-20-20 규칙처럼 눈의 피로를 줄이는 습관.
- 가벼운 진통제 사용: 일반 의약품으로 알려진 아세트아미노펜(타이레

In [ ]:
# user_prompt가 question보다 먼저 작성됐을 경우 -> 에러발생(question_ 이라는 변수에 할당된 내용이 없기때문에)

user_prompt = f"""\
아래의 사용자 질의에 작성된 통증에 따라 가능성 있는 원인과 해결 방법을 작성해주세요.

사용자 질의: {question_}
"""

question_ = "머리가 아파요ㅠㅠ"

response = model.invoke(user_prompt)
print(response.content)

NameError: name 'question_' is not defined

## 2.2 ChatPromptTemplate 활용

LangChain에서 지원하는 Template 중 ChatPromptTemplate을 활용하여 prompt를 완성해보자

In [1]:
from langchain_core.prompts import ChatPromptTemplate

# Template에 전달할 user prompt 작성
user_prompt = """\
아래의 사용자 질의에 작성된 통증에 따라 가능성 있는 원인과 해결 방법을 작성해주세요.

사용자 질의: {question}
"""

# Template 정의
# 딕셔너리가 아닌 튜플이니까 별도로 키값없이 role 과 content을 순서대로 입력, 
template = ChatPromptTemplate([
        ("user", user_prompt)
    ])

# template에 사용자 입력을 전달하여 최종 prompt 완성하기
final_prompt = template.invoke({"question": "머리가 너무 아파요 ㅠㅠ"})

# 모델에 전달될 최종 프롬프트 형태 확인
print(f"모델에 최종적으로 전달될 프롬프트 형태:")
final_prompt

모델에 최종적으로 전달될 프롬프트 형태:


ChatPromptValue(messages=[HumanMessage(content='아래의 사용자 질의에 작성된 통증에 따라 가능성 있는 원인과 해결 방법을 작성해주세요.\n\n사용자 질의: 머리가 너무 아파요 ㅠㅠ\n', additional_kwargs={}, response_metadata={})])

In [2]:
print(final_prompt.messages[0].content)

아래의 사용자 질의에 작성된 통증에 따라 가능성 있는 원인과 해결 방법을 작성해주세요.

사용자 질의: 머리가 너무 아파요 ㅠㅠ



In [3]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5-nano")

# 최종 프롬프트(final_prompt)를 모델에 전달하여 응답 생성해보기
response = model.invoke(final_prompt)
print(response.content) 

아프다니 정말 속상하시겠어요. 두통의 가능 원인과 스스로 시도해볼 수 있는 관리 방법을 정리해 드릴게요. 다만 증상이 심하거나 낯선 경우에는 병원에서 진료받는 것이 안전합니다.

1) 자주 생기는 두통의 가능 원인
- 긴장성 두통: strain된 자세나 스트레스, 피로로 양쪽 머리에 이런저런 두통이 올 때.
- 편두통: 한쪽이 심하게 아프고, 빛과 소리에 민감하거나 메스꺼움/구토가 동반될 때.
- 부비동 두통: 얼굴 앞부분이나 이마 쪽 압박감이 있고 코막힘/콧물 동반.
- 카페인 관련: 카페인을 갑자기 줄이거나 너무 많이 마신 뒤 생길 수 있음.
- 약물 과용두통: 진통제를 자주 쓰면 자꾸 더 아파질 때.
- 안구 피로/시력 부담: 컴퓨터 화면 오래 보면 생길 수 있음.
- 탈수나 수면 부족: 수분 부족과 수면 부족이 흔한 원인.
- 호르몬 변화나 임신 등 생리주기 관련 요인.
- 드물지만 심각한 경우 신경학적 이상이나 뇌질환의 전구 신호일 수 있음(다음의 신호에 주의).

2) 즉시 병원 진료가 필요할 수 있는 빨간 깃발(응급 징후)
즉시 의사 상담이 필요하거나 응급실 방문이 권합니다.
- 갑작스럽게 너무 심한 두통이 시작되었을 때(천둥처럼 느껴지는 통증)
- 의식 소실, 말하기 어려움, 한쪽 팔다리 마비나 감각 저하
- 심한 목 뻣뻣함, 열이 동반된 고열
- 구토가 지속되거나 의식 변化가 동반될 때
- 최근 머리 부상(심한 충격 없이 시작된 두통 포함)
- 기존 질환이 많거나 임신 중이거나 면역저하 상태인 경우
- 지속적으로 1-2주 이상 지속되거나 매일 아프고 일상생활에 지장

3) 집에서 시도해볼 수 있는 관리 방법
- 충분한 수분 섭취와 규칙적인 식사
- 휴식이 가능한 조용하고 어두운 환경에서 쉬기
- 냉찜질은 편두통에, 온찜질은 근긴장성 두통에 도움이 될 수 있음
- 자세 교정과 간단한 얼굴/목 스트레칭으로 긴장을 풀기
- 카페인 섭취를 현재 상태에 맞춰 조절(과다 섭취 또는 갑작스런 중단 피하기)
- 수면 패턴 안정화(정해진 수면 시간 지키기)
-

In [4]:
# system에도 전달해야 할 변수가 있을 경우

# 1. Template을 만들기 위한 system, user prompt 정의
system_prompt = "사용자가 입력한 영어를 {language}로 번역하세요."
user_prompt = "다음은 사용자의 입력입니다: {question}"

# 2. template 정의
template = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", user_prompt)
    ])

# 3. template에 사용자 입력을 전달하여 최종 prompt 완성하기
final_prompt = template.invoke({"language": "Korean", "question": "Where are you from?"})
final_prompt

ChatPromptValue(messages=[SystemMessage(content='사용자가 입력한 영어를 Korean로 번역하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='다음은 사용자의 입력입니다: Where are you from?', additional_kwargs={}, response_metadata={})])

In [5]:
# final_prompt 객체 형태 확인 
"""
ChatPromptValue(
    messages=[
        SystemMessage(
            content='사용자가 입력한 영어를 Korean로 번역하시오.', 
            additional_kwargs={}, 
            response_metadata={}
        ), 
        HumanMessage(
            content='다음은 사용자의 입력입니다: Where are you from?', 
            additional_kwargs={}, 
            response_metadata={}
        )
    ]
)
"""

"\nChatPromptValue(\n    messages=[\n        SystemMessage(\n            content='사용자가 입력한 영어를 Korean로 번역하시오.', \n            additional_kwargs={}, \n            response_metadata={}\n        ), \n        HumanMessage(\n            content='다음은 사용자의 입력입니다: Where are you from?', \n            additional_kwargs={}, \n            response_metadata={}\n        )\n    ]\n)\n"

In [6]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5-nano")

# 최종 프롬프트(final_prompt)를 모델에 전달하여 응답 생성해보기
response = model.invoke(final_prompt)
print(response.content) 

어디에서 왔어요?


# 3. LCEL (LangChain Expression Language)

- Runnable이란? 
    - Runnable 객체는 주로 Java 언어에서 멀티스레딩(다중 작업 처리)을 위해 사용되는 인터페이스
    - Python에서도 일부 프레임워크(LangChain과 같은)에서 유사한 개념이 사용됨 
- LangChain에서의 Runnable
    - 함수형 실행 단위를 래핑한 객체로, 체인으로 연결된 함수들을 캡슐화한(하나의 객체로 생성) 실행 가능한 객체를 의미 
    - 하나의 입력과 하나의 출력으로 실행 가능한 함수형태를 의미하기도 함
    - LangChain에서는 .invoke(), .stream(), .batch()와 같은 동일한 인터페이스를 가진 함수형태를 Runnable 객체라고 부르기도 함 
    - 따라서 Runnable 객체는 .invoke(), .stream(), .batch() 등의 메서드로 실행된다고 보면 됨
- 2번에서 정의한 template객체와 model객체도 Runnable 객체의 일종
- Runnabel 객체들을 chain으로 엮어 하나의 Runnable로 만들어보자 
    - 파이프 연산자: 여러개의 Runnable을 `|`(파이프) 연산자로 연결하면 자동으로 한 개의 Runnable 객체가 생성됨 
    - `|`(파이프) 기호의 역할
        - 서로 다른 구성요소(Runnable객체) 연결
        - 한 구성요소의 출력을 다음 구성요소의 입력으로 전달 
        - 즉, 사용자 입력이 템플릿으로 전달된 후 템플릿의 출력이 model로 전달 됨 
        - 각 구성요소를 개별적으로 살펴보면 어떻게 실행되는지 이해할 수 있음

In [4]:
# 두 개의 Runnable 객체를 파이프로 연결하여 한 개의 Runnable 객체(chain) 생성해보기
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser # 출력값을 우리가 원하는 값으로만 파싱해주는 클래스.
from langchain_openai import ChatOpenAI

# 1.Template에 전달할 system, user prompt 작성
system_prompt = "사용자가 입력한 영어를 {language}로 번역하세요."
user_prompt = "다음은 사용자의 입력입니다: {question}"

# 2.첫 번째 runnable객체(template) 생성
runnable1 = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", user_prompt)
])

# 3.두 번째 runnable객체(model) 생성
runnable2 = ChatOpenAI(model="gpt-5-nano", temperature=0.2)

# 4.최종 runnable객체(chain) 생성
chain = runnable1 | runnable2 | StrOutputParser() # stroutputparser: str값만 빼서 반환해줌 -> print(response.content) content객체로 출력할필요 없음

response = chain.invoke({"language": "Korean", "question": "where are you from?"})
print(response)

당신은 어디에서 왔나요?


# 4. 실습1

주어진 system과 question으로 chain을 생성하여 모델의 답변을 생성해보자


In [18]:
system = """\
당신은 금융 상담사입니다.
고객 정보를 분석해서 추천 상품 3개를 제시하되, 각 상품마다 "예상 수익률"과 "리스크 수준"을 명시하세요.
추가로, 각 상품마다 한 줄로 장점을 설명하세요.

출력 형식:
상품1: [상품명] - 수익률 X%, 리스크 [상/중/하]
상품2: [상품명] - 수익률 X%, 리스크 [상/중/하]  
상품3: [상품명] - 수익률 X%, 리스크 [상/중/하]\
"""

question = "월급 300만원, 저축 100만원 하고 있어요. 재테크 방법 알려주세요"

In [11]:
# 세은이 답변
# 1.Template에 전달할 system, user prompt 작성
system_prompt = "사용자가 입력한 정보를 분석해서 금융상품 3개를 예상 수익률과 리스크 수준을 명시하여 추천하세요."
user_prompt = "다음은 사용자의 입력입니다: {question}"

# 2.첫 번째 runnable객체(template) 생성
runnable1 = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", user_prompt)
])

# 3.두 번째 runnable객체(model) 생성
runnable2 = ChatOpenAI(model="gpt-5-nano", temperature=0.2)

# 4.최종 runnable객체(chain) 생성
chain = runnable1 | runnable2 | StrOutputParser() # stroutputparser: str값만 빼서 반환해줌 -> print(response.content) content객체로 출력할필요 없음

response = chain.invoke({"system": "content", "question": "월급 300만원, 저축 100만원 하고 있어요. 재테크 방법 알려주세요"})
print(response)


좋아요. 주신 정보로 현실적으로 시작하기 쉬운 3가지 금융상품을, 예상 수익률과 리스크 수준과 함께 제안드릴게요. 참고로 수익은 과거가 항상 미래를 보장하지 않으며, 투자 기간과 금리/시장 상황에 따라 달라집니다.

1) 상품 A: 국채형 ETF/펀드 (저위험)
- 예시: 국채형 ETF 또는 단기 국채 펀드
- 예상 수익률(연간): 약 2%대 초반 ~ 3%대 중반
- 리스크 수준: 낮음
- 특징 및 적합도: 원금손실 위험이 거의 없거나 매우 낮은 편에 가까운 안정자산
- 투자 방식 제안: 매달 100만원 저축 중 일정 비율을 이쪽으로 안정적으로 분할투자
- 권장 공간: 자금의 안정성 우선 시 30-60%를 여기에 배분해도 좋습니다.

2) 상품 B: 혼합형 자산배분 펀드 (균형형)
- 예시: 주식과 채권이 섞인 자산배분 펀드(예: 주식 비중 40–60%, 채권 비중 40–60% 정도의 운용 구조)
- 예상 수익률(연간): 약 3%대 중후반 ~ 5%대 초반
- 리스크 수준: 중간
- 특징 및 적합도: 주식과 채권의 밸런스로 비교적 안정적이면서도 성장 가능성도 추구
- 투자 방식 제안: 장기 목표가 있고 변동성을 감내할 수 있을 때 유용. 매달 100만원 중 일부 분할투자에 적합
- 권장 공간: 자산의 20–50%를 이 쪽으로 배분해도 균형 잡힌 포트가 됩니다.

3) 상품 C: 주식형 ETF/펀드 (대형주 또는 글로벌 주식)
- 예시: 코스피200 ETF나 글로벌 대형주 ETF(S&P 500 트래킹 등)
- 예상 수익률(연간): 약 5%대 중후반 ~ 8%대 초반(장기 기준)
- 리스크 수준: 높음
- 특징 및 적합도: 고성장을 기대하지만 단기 변동성이 큼. 장기 투자에 적합
- 투자 방식 제안: 원금의 큰 비중을 주식형으로 두되, 포트의 변동성을 견딜 수 있는 사람에게 적합
- 권장 공간: 자산의 10–40% 정도를 이 쪽으로 배분하는 것이 무난합니다.

실전 운영 가이드(초보자용 기본 틀)
- 먼저 긴급자금 마련: 생활비 3~6개월치를 현금성 자산으로 확보하는

In [16]:
system = """\
당신은 금융 상담사입니다.
고객 정보를 분석해서 추천 상품 3개를 제시하되, 각 상품마다 "예상 수익률"과 "리스크 수준"을 명시하세요.
추가로, 각 상품마다 한 줄로 장점을 설명하세요.

출력 형식:
상품1: [상품명] - 수익률 X%, 리스크 [상/중/하]
상품2: [상품명] - 수익률 X%, 리스크 [상/중/하]  
상품3: [상품명] - 수익률 X%, 리스크 [상/중/하]\
"""

question = "월급 300만원, 저축 100만원 하고 있어요. 재테크 방법 알려주세요"

In [17]:
# 선생님 답안 (순서가 중요!)

# runnable1: template 생성
template = ChatPromptTemplate([
    ("system", system),
    ("user", "{question}")
])

# runnable2: model 생성
model = ChatOpenAI(model="gpt-5-nano")

# chain 생성
chain = template | model | StrOutputParser()

# 응답 생성
response = chain.invoke(question)
print(response)

상품1: [정기예금(저축성)] - 수익률 2.5%, 리스크 하 - 장점: 원금 보장에 가까워 비상자금으로 안정적 운영

상품2: [혼합형 주식-채권 펀드] - 수익률 6%, 리스크 중 - 장점: 자산 배분으로 변동성 완화와 중장기 성장 기회

상품3: [해외 주식형 ETF(미국 대형주)] - 수익률 9%, 리스크 상 - 장점: 글로벌 성장 기회에 투자


# 5. 실습2

주어진 system과 question으로 chain을 생성하여 모델의 답변을 생성해보자

In [2]:
system = """\
당신은 은행 고객센터 AI 분류 시스템입니다.

고객 문의를 다음 카테고리 중 하나로 정확히 분류하세요:
{categories}

추가 설명 없이 분류명만 출력하세요. 
"""

categories = {
    "account_inquiry": "계좌 잔액, 거래 내역 조회 문의",
    "card_issue": "카드 분실, 도난, 재발급 문의",
    "loan_application": "대출 신청 또는 대출 가능 여부 문의",
    "interest_rate": "예금/적금 금리, 대출 이자율 문의",
    "transfer": "계좌 이체, 송금 관련 문의",
    "limit_change": "카드 한도, 이체 한도 변경 요청",
    "payment_issue": "카드값 결제, 연체 관련 문의",
    "fraud_report": "이상 거래, 사기 의심 신고",
    "investment": "펀드, 주식, 금융 상품 투자 문의",
    "branch_atm": "영업점, ATM 위치 및 영업시간 문의",
    "mobile_app": "모바일 뱅킹 앱 사용 관련 문의",
    "complaint": "불만 사항 또는 개선 요청",
    "other": "기타 문의"
}

question = "카드를 잃어버렸어요. 재발급 받으려면 어떻게 해야 하나요?"

In [ ]:

# runnable1: template 생성
template = ChatPromptTemplate([
    ("system", system),
    ("user", "{question}")
])

# runnable2: model 생성
model = ChatOpenAI(model="gpt-5-nano")

# chain 생성
chain = template | model | StrOutputParser()

# 응답생성
response = chain.invoke({"categories": categories, "question": question})
print(response)

card_issue


## 5.1 다양한 고객 문의 분류

In [6]:
system = """\
당신은 은행 고객센터 AI 분류 시스템입니다.

고객 문의를 다음 카테고리 중 하나로 정확히 분류하세요:
{categories}

추가 설명 없이 분류명만 출력하세요. 
"""

categories = {
    "account_inquiry": "계좌 잔액, 거래 내역 조회 문의",
    "card_issue": "카드 분실, 도난, 재발급 문의",
    "loan_application": "대출 신청 또는 대출 가능 여부 문의",
    "interest_rate": "예금/적금 금리, 대출 이자율 문의",
    "transfer": "계좌 이체, 송금 관련 문의",
    "limit_change": "카드 한도, 이체 한도 변경 요청",
    "payment_issue": "카드값 결제, 연체 관련 문의",
    "fraud_report": "이상 거래, 사기 의심 신고",
    "investment": "펀드, 주식, 금융 상품 투자 문의",
    "branch_atm": "영업점, ATM 위치 및 영업시간 문의",
    "mobile_app": "모바일 뱅킹 앱 사용 관련 문의",
    "complaint": "불만 사항 또는 개선 요청",
    "other": "기타 문의"
}

# 다양한 금융 케이스들
finance_cases = [
    "통장 잔고가 얼마인지 확인하고 싶어요",
    "집을 사려고 하는데 대출 받을 수 있나요?",
    "모르는 곳에서 카드 겔제 문자가 왔어요",
    "적금 금리가 어떻게 되나요?"
]

In [8]:
for case in finance_cases:
    template = ChatPromptTemplate([
        ("system", system),
        ("user", "{case}")
    ])

    model = ChatOpenAI(model="gpt-5-nano")

    chain = template | model | StrOutputParser()

    response = chain.invoke({"categories": categories, "case": case })
    print(f"문의: {case}")
    print(f"-> 분류: {response}\n")

문의: 통장 잔고가 얼마인지 확인하고 싶어요
-> 분류: account_inquiry

문의: 집을 사려고 하는데 대출 받을 수 있나요?
-> 분류: loan_application

문의: 모르는 곳에서 카드 겔제 문자가 왔어요
-> 분류: fraud_report

문의: 적금 금리가 어떻게 되나요?
-> 분류: interest_rate



## 5.2 분류 결과에 따른 자동 응답

In [11]:
# 분류 결과에 따른 자동 응답 함수 정의
def auto_response_finance(input_data):
    """금융 문의 분류 후 자동 응답 반환"""

    # 1. 프롬프트 정의
    system = """\
당신은 은행 고객센터 AI 분류 시스템입니다.

고객 문의를 다음 카테고리 중 하나로 정확히 분류하세요:
{categories}

추가 설명 없이 분류명만 출력하세요.\
"""

    categories = {
        "account_inquiry": "계좌 잔액, 거래 내역 조회 문의",
        "card_issue": "카드 분실, 도난, 재발급 문의",
        "loan_application": "대출 신청 또는 대출 가능 여부 문의",
        "interest_rate": "예금/적금 금리, 대출 이자율 문의",
        "transfer": "계좌 이체, 송금 관련 문의",
        "limit_change": "카드 한도, 이체 한도 변경 요청",
        "payment_issue": "카드값 결제, 연체 관련 문의",
        "fraud_report": "이상 거래, 사기 의심 신고",
        "investment": "펀드, 주식, 금융 상품 투자 문의",
        "branch_atm": "영업점, ATM 위치 및 영업시간 문의",
        "mobile_app": "모바일 뱅킹 앱 사용 관련 문의",
        "complaint": "불만 사항 또는 개선 요청",
        "other": "기타 문의"
    }

    # 2. 의도 분류
    template = ChatPromptTemplate([
        ("system", system),
        ("user", "{input_data}")
    ])

    model = ChatOpenAI(model="gpt-5-nano")
    
    chain = template | model | StrOutputParser()

    res_intent = chain.invoke({"categories": categories, "input_data": input_data})

    # 3. 의도별 맞춤 응답
    res_templates = {
        "card_issue": """\
카드 재발급 안내드립니다.

📞즉시 카드 정지: 1588-xxxx (24시간)
💳재발급 신청: 모바일 앱 또는 영업점 방문
⏱️발급 기간: 영업일 기준 3-5일
💸발급 수수로: 무료

추가 도움이 필요하시다면 말씀해주세요.\
        """,
        "account_inquiry":"""\
계좌 조회 방법 안내드립니다.

📱모바일 앱: [은행명] 앱 > 조회 메뉴
💻인터넷 뱅킹: www.bank.com
🏧ATM: 전국 ATM에서 조회 가능 
📞전화: 1588-xxxx > ARS 1번

실시간 조회가 가능합니다.\
        """
    }

    auto_reply = res_templates.get(res_intent, "상담원 연결이 필요합니다. 잠시만 기다려주세요.")

    print(f"고객 문의: {input_data}")
    print(f"분류 결과: {res_intent}")
    print(f"자동 응답: {auto_reply}")

In [12]:
# 결과 확인
result = auto_response_finance("카드를 잃어버렸어요")
result

고객 문의: 카드를 잃어버렸어요
분류 결과: card_issue
자동 응답: 카드 재발급 안내드립니다.

📞즉시 카드 정지: 1588-xxxx (24시간)
💳재발급 신청: 모바일 앱 또는 영업점 방문
⏱️발급 기간: 영업일 기준 3-5일
💸발급 수수로: 무료

추가 도움이 필요하시다면 말씀해주세요.        


# 6. RunnableParallel

- Runnable을 동시에 병렬로 실행할 수 있도록 해주는 조합
- 이 때 모든 Runnable에게는 동일한 input이 전달되며, 출력은 각 Runnable에 따른 결과 각각이 딕셔너리로 묶여 출력됨

## 6.1 예제1

In [2]:
# 한국어를 입력하면 일어와 영어 번역을 동시에 출력할 수 있는 병렬 Runnable을 구현해보자
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# 두 개의 template 생성
template1 = ChatPromptTemplate([
    ("system", "당신은 유능한 번역가입니다. 입력된 한국어를 영어로 번역하세요."),
    ("user", "{question}")
])

template2 = ChatPromptTemplate([
    ("system", "당신은 유능한 번역가입니다. 입력된 한국어를 일어로 번역하세요."),
    ("user", "{question}")
])

# 모델 정의
model_gpt = ChatOpenAI(model="gpt-5-nano")
model_gmni = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

# 두 개의 chain(runnable객체) 생성
chain1 = template1 | model_gpt | StrOutputParser()
chain2 = template2 | model_gmni | StrOutputParser()

# 두 개의 runnable(chain1, chain2)을 하나의 runnable객체로 묶기(병렬 실행)
runnable_parallel = RunnableParallel({
    "영어 번역": chain1,
    "일어 번역": chain2
})

final_output = runnable_parallel.invoke("몇살이세요?")
final_output

{'영어 번역': 'How old are you?', '일어 번역': '몇 살이세요?'}

## 6.2 예제2

- 투자 포트폴리오 다각도 분석
    - 포트폴리오: 투자자가 보유한 여러 자산의 구성 비율
    - 투자 전략은 개별 종목이 아니라 포트폴리오 전체 구조를 기준으로 판단 
    

In [4]:
from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# 보수적 투자 전략 프롬프트
template_conservative = ChatPromptTemplate([
    ("system", "당신은 보수적 투자 전략 전문가입니다. 포트폴리오를 안정성 관점에서 분석하고 개선안을 제시하세요.(5줄 이내)"),
    ("user", "{input_data}")
])

# 공격적 투자 전략 프롬프트
template_aggressive = ChatPromptTemplate([
    ("system", "당신은 공격적 투자 전략 전문가입니다. 포트폴리오를 수익률 관점에서 분석하고 개선안을 제시하세요.(5줄 이내)"),
    ("user", "{input_data}")
])

# 균형적 투자 전략 프롬프트
template_balanced = ChatPromptTemplate([
    ("system", "당신은 균형 투자 전략 전문가입니다. 포트폴리오를 리스크-수익 균형 관점에서 분석하고 개선안을 제시하세요.(5줄 이내)"),
    ("user", "{input_data}")
])

# 모델 정의
model = ChatOpenAI(model="gpt-5-nano", temperature=0)

# 3개의 chain 생성 
chain_conversaitve = template_conservative | model | StrOutputParser()
chain_aggressive = template_aggressive | model | StrOutputParser()
chain_balanced = template_balanced | model | StrOutputParser()

# 병렬 runnable 생성
runnable_portfolio = RunnableParallel({
    "보수형 전략": chain_conversaitve,
    "공격형 전략": chain_aggressive,
    "균형 전략": chain_balanced
})

# input_data 작성
portfolio = "현재 포트폴리오: 국내주식 70%, 해외주식 20%, 현금 10%"
result_portfolio = runnable_portfolio.invoke(portfolio)

print(f"\n{portfolio}\n")
print(f"[보수형 전략 분석]\n{result_portfolio['보수형 전략']}\n")
print(f"[공격형 전략 분석]\n{result_portfolio['공격형 전략']}\n")
print(f"[균형 전략 분석]\n{result_portfolio['균형 전략']}\n")


현재 포트폴리오: 국내주식 70%, 해외주식 20%, 현금 10%

[보수형 전략 분석]
- 진단: 국내주식 70%는 과다한 주식 편중으로 변동성 및 하방 리스크가 큼.  
- 개선 방향: 주식 비중을 40-50%로 축소하고 채권/현금 비중을 50-60%로 늘려 안정성 강화.  
- 제안 구성 예시: 국내주식 30%, 해외주식 15-20%, 채권 35-45%, 현금 5-10%.  
- 채권 구성: 국내외 국채와 단기채로 만기 다변화, 금리 리스크 관리에 중점.  
- 리밸런싱: 연 1회 또는 자산 편차 5% 이상 시 리밸런싱, 필요 시 해외주식 환헤지 고려.

[공격형 전략 분석]
- 현재 포트는 국내주식 70%, 해외주식 20%, 현금 10%로 국내 집중과 현금 보유로 수익률 변동성이 큼. 
- 공격적 관점에서 해외 비중 확대와 성장/소형주 노출 증가를 권고. 
- 권고 분배 예시: 국내 40%, 해외 50%, 현금 10%로 글로벌 다변화와 고성장 노출 강화. 
- 해외 구성은 신흥시장/소형주 비중을 확대하는 것을 목표로 하되 전체 위험 관리도 병행. 
- 연 1회 리밸런싱과 변동성 급등 시 트리거 설정으로 규칙적으로 관리.

[균형 전략 분석]
- 진단: 국내주식 70%로 리스크 집중, 해외 20%/현금 10%는 다변화 효과가 미미합니다.  
- 목표: 주식 비중을 50% 전후로 낮추고, 채권/현금 비중을 합쳐 50%로 조정해 리스크-수익 균형 추구.  
- 제안 예시: 국내주식 25%, 해외주식 25%, 채권 40%, 현금 10%.  
- 해외주식은 지역 분산을 강화하고 비용 효율은 글로벌 ETF로, 필요시 달러-원 헤지 고려.  
- 관리: 연 1–2회 리밸런싱으로 목표 비중 유지, 유동성/비용과 헤지 여부를 점검.



In [5]:
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# 보수적 투자 전략 프롬프트
template_conservative = ChatPromptTemplate([
    ("system", "당신은 보수적 투자 전략 전문가입니다. 포트폴리오를 안정성 관점에서 분석하고 개선안을 제시하세요.(5줄 이내)"),
    ("user", "{input_data}")
])

# 공격적 투자 전략 프롬프트
template_aggressive = ChatPromptTemplate([
    ("system", "당신은 공격적 투자 전략 전문가입니다. 포트폴리오를 수익률 관점에서 분석하고 개선안을 제시하세요.(5줄 이내)"),
    ("user", "{input_data}")
])

# 균형적 투자 전략 프롬프트
template_balanced = ChatPromptTemplate([
    ("system", "당신은 균형 투자 전략 전문가입니다. 포트폴리오를 리스크-수익 균형 관점에서 분석하고 개선안을 제시하세요.(5줄 이내)"),
    ("user", "{input_data}")
])

# 모델 정의
model_gmni = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

# 3개의 chain 생성 
chain_conversaitve = template_conservative | model | StrOutputParser()
chain_aggressive = template_aggressive | model | StrOutputParser()
chain_balanced = template_balanced | model | StrOutputParser()

# 병렬 runnable 생성
runnable_portfolio = RunnableParallel({
    "보수형 전략": chain_conversaitve,
    "공격형 전략": chain_aggressive,
    "균형 전략": chain_balanced
})

# input_data 작성
portfolio = "현재 포트폴리오: 국내주식 70%, 해외주식 20%, 현금 10%"
result_portfolio = runnable_portfolio.invoke(portfolio)

print(f"\n{portfolio}\n")
print(f"[보수형 전략 분석]\n{result_portfolio['보수형 전략']}\n")
print(f"[공격형 전략 분석]\n{result_portfolio['공격형 전략']}\n")
print(f"[균형 전략 분석]\n{result_portfolio['균형 전략']}\n")


현재 포트폴리오: 국내주식 70%, 해외주식 20%, 현금 10%

[보수형 전략 분석]
- 현재 포트는 국내주식 70%로 주식 편중이 심해 변동성 및 리스크가 큽니다. 
- 안정성을 높이려면 주식 비중을 축소하고 채권/현금을 늘리는 것이 바람직합니다. 
- 권장 목표 예시: 국내주식 30%, 해외주식 15%, 채권 45%, 현금 10%. 
- 채권은 국채/우량회사채를 혼합한 분산 ETF/펀드로 구성하고 해외주식은 축소해 총리스크를 낮추세요. 
- 연 1회 리밸런싱 또는 시장 변동성 5-10% 초과 시 재조정하고 비상 현금은 5-10% 정도 유지하십시오.

[공격형 전략 분석]
- 진단: 현재 포트는 국내주식 70%, 해외주식 20%, 현금 10%로 고수익 추구 성향이나 국내 집중과 현금 드래그가 위험/수익성에 제약.

- 개선 방향: 글로벌 다변화와 현금의 기회비용 최소화를 통해 평균 수익률을 끌어올리는 구조로 전환.

- 재배치 예시: 국내 비중을 40-50%로 축소하고 해외를 40-45%로 확대, 현금은 5-10%로 유지(필요 시 재투자 대기금으로 활용).

- 추가 다변화: 신흥시장 5-10% 및 스타일(성장/가치)과 지역 다변화, 대형+소형 혼합, REIT/원자재 같은 대체자산 0-10%를 소량 포함.

- 실행 팁: 저비용 ETF로 구성하고 연 1회 이상 리밸런싱하여 드리프트를 관리하고, 현금은 전략적 기회에만 활용.

[균형 전략 분석]
- 분석: 국내주식 70%, 해외주식 20%, 현금 10%는 주식 편중이 크고 국내 리스크에 더 민감합니다. 다변화가 부족해 변동성 대비 방어력이 낮을 수 있습니다.
- 개선 포인트: 주식 비중을 낮추고 채권·대체자산으로 포트의 안정성과 수익-리스크 균형을 높이는 것이 좋습니다.
- 제안 포트 예: 국내주식 30%, 해외주식 25%, 채권 25%, 현금 0-5%, 대체자산 15-20%.
- 실행 팁: 비용 효율적 ETF/인덱스 활용, 연 1회 리밸런싱 또는 시장 변화에 맞춘 점진적 조정으로 위험 관리.
- 주의:

## 6.3 예제3

투자 전략 결과 비교 및 최적안 선택


In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

def select_best_strategy(portfolio_data):
    """3가지 전략을 생성하고 상황에 맞는 최적 전략 추천"""

    # 병렬로 3가지 전략 생성
    strategies = runnable_portfolio.invoke(portfolio_data)

    # 최적 전략 프롬프트
    user_prompt = """\
포트폴리오 : {portfolio}

보수형 전략: {conservative}
공격형 전략: {aggressive}
균형 전략: {balanced}\
    """

    selector_template = ChatPromptTemplate([
        ("system", "당신은 투자 전략 컨설턴트입니다. 고객 포트폴리오와 3가지 전략 분석을 보고 가장 적합한 전략 1개를 선택하세요."),
        ("user", user_prompt)
    ])

    # model 정의 
    model = ChatOpenAI(model="gpt-5-nano")

    # chain 생성
    selector_chain = selector_template | model | StrOutputParser()

    # 최종 응답 생성
    final_choice = selector_chain.invoke({
        "portfolio": portfolio,
        "conservative": strategies["보수형 전략"],   # strategies의 결과값이 {}딕셔너리니까 키값으로 접근해서 밸류값 추출.
        "aggressive": strategies["공격형 전략"],
        "balanced": strategies["균형 전략"]
    })

    return strategies, final_choice

In [19]:
portfolio = "국내주식 70%, 해외주식 20%, 현금 10%"
strategies, final_choice = select_best_strategy(portfolio)

print("\n고객 정보: {portfolio}\n")
print(f"[추천 전략]\n{final_choice}")


고객 정보: {portfolio}

[추천 전략]
권장 전략: 균형 전략

이유 요약
- 현재 포트는 국내주식 편중(70%)과 해외주식(20%)의 비중 탓에 환리스크 및 변동성 부담이 큽니다. 
- 균형 전략은 주식 비중을 60–65%로 낮추고 채권 등 고정수입 자산을 25–35%로 확보하여 방어력을 강화합니다. 또한 물가연동채를 포함한 채권 구성으로 인플레이션 리스크도 대비합니다.
- 해외주식은 지역 다변화와 필요 시 환헤지를 고려하도록 하여 글로벌 다변화를 강화합니다.
- 현금 비중은 10%로 유지하되, 정기 재조정(리밸런싱) 규칙을 통해 목표 위험 허용치에 맞춘 안정적인 glide path를 유지합니다.

권장 목표 자산배분 (포트폴리오 100%)
- 국내주식: 32%
- 해외주식: 28%
- 채권: 30% (국내/해외 혼합 및 물가연동채 포함)
- 현금: 10%

구현 가이드
- 실행 방식: 저비용 인덱스/ETF 중심으로 구성하고, 가능하면 해외주식은 지역 다변화(선진/신흥시장) 및 필요 시 환헤지 옵션 사용.
- 채권 구성: 국내 채권과 글로벌 채권을 혼합하고, 인플레이션에 대응하는 물가연동채(TIPS 등) 포함으로 인플레이션 리스크를 완화.
- 해외주식 관리: 지역 다변화와 함께 통화 리스크 관리(필요 시 환헤지 고려). 장기적으로는 글로벌 ETF로 재구성.
- 리밸런싱 규칙: 분기 또는 반기마다 현 포트와 목표 배분 간 차이가 일정 임계치에 도달하면 조정. 비용, 세금, 거래창출을 고려한 실용적 규칙 적용.
- 리스크 관리 추가: 시장 하락 시 현금 여력과 채권 비중이 방어 역할을 하도록 설정. 필요 시 현금을 유연성 자금으로 활용하는 것도 고려.
- 실행 예시: 지금의 포트(국내 70%, 해외 20%, 현금 10%)를 위 목표로 맞추려면,
  - 국내주식 비중을 약 20%포인트 낮추고 채권으로 약 20%포인트, 해외주식으로 약 8%포인트 추가 확보
  - 남은 현금은 10% 그대로 두되, 재투자 시점에 분할 매수로 리밸런싱
  - 해외주식은 2

In [20]:
portfolio2 = "30대 직장인, 투자 경험 2년, 국내주식 80%, 현금20%"
strategies, final_choice = select_best_strategy(portfolio2)

print("\n고객 정보: {portfolio2}\n")
print(f"[추천 전략]\n{final_choice}")


고객 정보: {portfolio2}

[추천 전략]
선택 전략: 균형 전략 (Balanced Strategy)

이유 요약
- 현재 포트의 가장 큰 문제는 국내주식 집중(70%)으로 인한 하방 리스크와 변동성입니다. 균형 전략은 국내 주식의 비중을 낮추고 국제 다변화와 채권 대책을 통해 하방 리스크를 효과적으로 관리합니다.
- 국제주식 비중을 포함한 글로벌 다변화를 통해 수익원 다변화와 변동성 관리가 가능하며, 채권/현금의 비중 확대는 샤프 비율 개선에 기여합니다.
- 실행이 비교적 합리적이며, 저비용 패시브 ETF 중심으로 구현할 수 있어 비용 효율성도 확보됩니다.
- 현금 비중도 비교적 합리적 수준(5-10%)으로 유지하면서 필요시 유동성 확보를 가능하게 합니다.

권장 목표 배분 (균형 전략의 예시 범위 내에서 실행 가능한 구성)
- 국내주식: 30%
- 해외주식: 20%
- 채권: 40%
- 현금성자산: 10%

구체적 실행 계획
1) 현재 포트 조정 포인트
- 현재: 국내주식 70%, 해외주식 20%, 현금 10%
- 목표: 국내주식 30%, 해외주식 20%, 채권 40%, 현금 10%
- 실행 방식: 국내주식에서 40%를 매도하여 채권으로 전환(예시: 국내주식 매도 40%를 채권으로 배분). 이렇게 하면 최종 구성이 30%/20%/40%/10%으로 정렬됩니다. 채권은 국내 국채와 글로벌 채권을 혼합한 형태로 구성하는 것이 좋습니다.
2) 자산 클래스별 구체적 구성
- 국내주식: 적극적으로 줄이되 30%로 유지
- 해외주식: 현재 20%를 유지하거나 소폭 증가(필요 시 국제주식 비중을 20%로 고정)
- 채권: 40% 목표(구성 예: 국내 국채 20%, 글로벌 채권 20%의 혼합)
- 현금성자산: 10% 유지(단기예금/MMF 등으로 보전)
3) 리밸런싱 주기 및 방법
- 리밸런싱 주기: 분기별 또는 반기별 재조정 권장. 초기에는 6개월 내 한 번 점검 후, 6-12개월 주기로 정기적 재조정.
- 재조정 기준: 각 범주가 목표 비중에서 ±5% 이상 벗

# 7. 실습3

RunnableParallel을 통해 제품 리뷰 자동 분석 시스템을 개발해보자

[문제]
- 3가지 병렬 분석: 감정 분석 + 핵심 키워드 추출 + 개선점 도출   
    - 감정 분석: 리뷰를 긍정, 중립, 부정으로 분류
    - 핵심 키워드 추출: 리뷰에서 중요한 핵심 키워드(단어) 3개 추출 (ex. 배송, 품질, 가격, 디자인, 서비스 등)
    - 개선점 도출: 리뷰를 통해 제품 개선에 대한 필요성이 언급되면 제시

[출력 예시]
- 리뷰 1: 배송이 정말 빠르고 제품 품질도 만족스러워요! 재구매 의사 100%
  - 감정: positive
  - 키워드: 배송, 품질, 만족
  - 개선점: 없음

In [21]:
sample_product_reviews = [
    "배송이 정말 빠르고 제품 품질도 만족스러워요! 재구매 의사 100%",
    "가격 대비 별로예요. 사진이랑 다르게 왔어요. 실망",
    "디자인은 예쁜데 내구성이 약한 것 같아요. 한 달 쓰니까 고장남",
    "포장이 꼼꼼하고 고객센터 응대도 친절했습니다.",
    "배송은 느렸지만 제품은 만족해요. 다음엔 빠른 배송 옵션 써야겠어요.",
    "완전 최고! 친구들한테 추천했어요. 가성비 갑",
    "환불하고 싶은데 절차가 너무 복잡해요.",
    "보통이에요. 가격 생각하면 이 정도는 되는 것 같아요."
]

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

def select_best_strategy(portfolio_data):
    """리뷰를 3가지 테마로 분류하여 감정분석, 핵심키워드 추출, 개선점 도출"""

    # 병렬로 3가지 분석 생성
    strategies = runnable_portfolio.invoke(portfolio_data)

    # 최적 전략 프롬프트
    user_prompt = """\
리뷰 : {reviews}

감정 : {emotion}
키워드 : {keyword_3}
개선점 : {upgrade}\
    """

    selector_template = ChatPromptTemplate([
        ("system", "당신은 고객만족센터의 현실적인 상담사입니다.. 고객의 리뷰를 분석하여 핵심키워드를 3가지 추출하여 감정 분석을 진행하고 개선점을 도출하세요."
    "- positive\n -neutral\n -negative\n\n"
     "다른 말은 절대 출력하지 마세요."),
        ("user", user_prompt)
    ])

    # model 정의 
    model = ChatOpenAI(model="gpt-5-nano")

    # chain 생성
    selector_chain = selector_template | model | StrOutputParser()

    # 최종 응답 생성
    final_choice = selector_chain.invoke({
        "reviews": reviews,
        "emotion": strategies["감정"],   
        "keyword_3": strategies["키워드"],
        "upgrade": strategies["개선점"]
    })

    return strategies, final_choice

# 근데 여기서 감정분석에서 긍정, 부정, 중립으로 나눠서 출력하도록 정의하고
# 핵심키워드를 3가지만 추출하도록하고
# 개선점이 없으면 없음으로 있으면 개선점을 3줄 이하로 추출하도록 해야할것같은데!!!!
    print(f"감정: {input_data}")
    print(f"분류 결과: {res_intent}")
    print(f"자동 응답: {auto_reply}")

In [23]:
# 지피티한테 위의 코드로 명령해서 만든 코드
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# 모델정의
model = ChatOpenAI(model="gpt-5-nano",temperature=0.2)

# 감정분석 정의
emotion_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 리뷰 감정 분석기입니다.\n"
     "반드시 아래 3개 중 하나만 출력하세요.\n"
     "- positive\n- neutral\n- negative\n"
     "다른 말은 절대 출력하지 마세요."),
    ("user", "{review}")
])

# 핵심키워드 3개 추출
keyword_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 리뷰 핵심 키워드 추출기입니다.\n"
     "리뷰에서 가장 중요한 키워드 3개만 추출하세요.\n"
     "출력 형식: 키워드1, 키워드2, 키워드3\n"
     "다른 설명은 출력하지 마세요."),
    ("user", "{review}")
])

# 개선점 도출 
upgrade_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 제품 개선점 분석기입니다.\n"
     "- 개선점이 없으면: 없음\n"
     "- 개선점이 있으면: 최대 3줄 이내로 간결히 작성\n"
     "다른 말은 출력하지 마세요."),
    ("user", "{review}")
])

# 체인으로 붙이기
emotion_chain = emotion_prompt | model | StrOutputParser()
keyword_chain = keyword_prompt | model | StrOutputParser()
upgrade_chain = upgrade_prompt | model | StrOutputParser()

review_analyzer = RunnableParallel({
    "감정": emotion_chain,
    "키워드": keyword_chain,
    "개선점": upgrade_chain
})

# 리뷰분석
def analyze_review(review: str):
    result = review_analyzer.invoke({"review": review})
    return {
        "리뷰": review,
        "감정": result["감정"],
        "키워드": result["키워드"],
        "개선점": result["개선점"]
    }

# 출력: 사아실~~ 나는 if랑 for문 잘 못씀ㅠㅠㅠ 
if __name__ == "__main__":
    for i, r in enumerate(sample_product_reviews, 1):
        res = analyze_review(r)
        print(f"\n리뷰 {i}: {res['리뷰']}")
        print(f"- 감정: {res['감정']}")
        print(f"- 키워드: {res['키워드']}")
        print(f"- 개선점: {res['개선점']}")



리뷰 1: 배송이 정말 빠르고 제품 품질도 만족스러워요! 재구매 의사 100%
- 감정: positive
- 키워드: 배송, 품질, 재구매의사
- 개선점: 없음

리뷰 2: 가격 대비 별로예요. 사진이랑 다르게 왔어요. 실망
- 감정: negative
- 키워드: 가격 대비, 사진과 다름, 실망
- 개선점: - 가격 대비 품질/가치 개선
- 사진과 실물의 일치도 강화
- 설명·광고의 정확성 향상

리뷰 3: 디자인은 예쁜데 내구성이 약한 것 같아요. 한 달 쓰니까 고장남
- 감정: negative
- 키워드: 디자인, 내구성, 고장
- 개선점: - 내구성 강화: 충격·진동에 강한 재료 및 조립 품질 개선
- 부품 신뢰성 개선 및 제조 공정의 품질 관리 강화
- 애프터서비스 개선: 보증 기간 확대 및 신속한 수리/교환 정책 도입

리뷰 4: 포장이 꼼꼼하고 고객센터 응대도 친절했습니다.
- 감정: positive
- 키워드: 포장, 응대, 친절
- 개선점: 없음

리뷰 5: 배송은 느렸지만 제품은 만족해요. 다음엔 빠른 배송 옵션 써야겠어요.
- 감정: positive
- 키워드: 배송, 만족, 빠른 배송
- 개선점: - 배송 속도 개선 필요
- 빠른 배송 옵션의 이용 방법과 비용/소요 시간 정보를 명확히 제공
- 빠른 배송 옵션 선택 프로세스의 간소화 필요

리뷰 6: 완전 최고! 친구들한테 추천했어요. 가성비 갑
- 감정: positive
- 키워드: 가성비, 추천, 최고
- 개선점: 없음

리뷰 7: 환불하고 싶은데 절차가 너무 복잡해요.
- 감정: negative
- 키워드: 환불, 절차, 복잡
- 개선점: 환불 절차를 한 화면에서 간소화하고 필요한 정보만 받도록 개선  
진행 상태 자동 추적 및 알림(상태 변화 시 알림) 추가  
요건·소요 시간 명확화와 FAQ/고객센터 접점의 접근성 개선

리뷰 8: 보통이에요. 가격 생각하면 이 정도는 되는 것 같아요.
- 감정: neutral
- 키워드: 가격, 보통, 가성비
- 개선점: 없음


In [27]:
# 선생님 답안
sample_product_reviews = [
    "배송이 정말 빠르고 제품 품질도 만족스러워요! 재구매 의사 100%",
    "가격 대비 별로예요. 사진이랑 다르게 왔어요. 실망",
    "디자인은 예쁜데 내구성이 약한 것 같아요. 한 달 쓰니까 고장남",
    "포장이 꼼꼼하고 고객센터 응대도 친절했습니다.",
    "배송은 느렸지만 제품은 만족해요. 다음엔 빠른 배송 옵션 써야겠어요.",
    "완전 최고! 친구들한테 추천했어요. 가성비 갑",
    "환불하고 싶은데 절차가 너무 복잡해요.",
    "보통이에요. 가격 생각하면 이 정도는 되는 것 같아요."
]

# 감정분석 프롬프트 및 템플릿 작성
sentiment_system = """\
제품 리뷰를 입력받으면 감정을 분류하세요:
- positive: 긍정적
- neutral: 중립적
- negative: 부정적

감정만 출력하세요

# 출력 예시
positive
"""
sentiment_template = ChatPromptTemplate([
    ("system", sentiment_system),
    ("user", "{review}")
])

# 핵심 키워드 추출 프롬프트 및 템플릿 작성
keyword_system = """\
리뷰에서 핵심 키워드를 추출하세요.
배송, 품질, 가격, 디자인, 서비스 등 중요한 단어 3개를 쉽표로 구분하여 출력하세요.

# 출력 예시
배송, 품질, 만족
"""
keyword_template = ChatPromptTemplate([
    ("system", keyword_system),
    ("user", "{review}")
])

# 개선점 도출 프롬프트 및 템플릿 작성
improvement_system = """\
리뷰 내용을 보고 개선이 필요한 부분을 1개만 제시해주세요.
개선점이 없으면 "없음"이라고 출력하세요.

# 출력 예시
배송 속도 개선 필요
"""

improvement_template = ChatPromptTemplate([
    ("system", keyword_system),
    ("user", "{review}")
])


# 모델 정의
model = ChatOpenAI(model="gpt-5-nano")

# chain 생성
sentiment_chain = sentiment_template | model | StrOutputParser()
keyword_chain = keyword_template | model | StrOutputParser()
improvement_chain = improvement_template | model | StrOutputParser()

# 병렬 runnable 객체 생성
parallel_runnable = RunnableParallel(
    sentiment=sentiment_chain,
    keyword=keyword_chain,
    improvement=improvement_chain
)

# 리뷰 분석 실행
print("\n[제품 리뷰 자동 분석 결과]\n")
result_marketing = []
for i, review in enumerate(sample_product_reviews[:3], 1):
    output = parallel_runnable.invoke(review)
    result_marketing.append({
        "리뷰내용": review,
        **output  # **dict: dictionary unpacking  
    })

    print(f"리뷰 {i}: {review}")
    print(f"    -> 감정: {output['sentiment']}")
    print(f"    -> 키워드: {output['keyword']}")
    print(f"    -> 개선점: {output['improvement']}")


[제품 리뷰 자동 분석 결과]

리뷰 1: 배송이 정말 빠르고 제품 품질도 만족스러워요! 재구매 의사 100%
    -> 감정: positive
    -> 키워드: 배송, 품질, 만족
    -> 개선점: 배송, 품질, 재구매 의사
리뷰 2: 가격 대비 별로예요. 사진이랑 다르게 왔어요. 실망
    -> 감정: negative
    -> 키워드: 가격, 디자인, 품질
    -> 개선점: 가격, 품질, 디자인
리뷰 3: 디자인은 예쁜데 내구성이 약한 것 같아요. 한 달 쓰니까 고장남
    -> 감정: negative
    -> 키워드: 디자인, 품질, 고장
    -> 개선점: 디자인, 품질, 만족


In [34]:
# 딕셔너리 언패킹 예시

output = {
    "sentiment": "positive",
    "keyword": "배송, 품질, 만족",
    "improvement": "없음"
}

# 언패킹 하지 않았을 경우
result1 = {
    "리뷰내용": "배송이 정말 빠르고 제품이 만족스러워요",
} 

result1["output"] = output  


# 언패킹 했을 경우
result2 = {
    "리뷰내용": "배송이 정말 빠르고 제품이 만족스러워요",
    **output
}

print(result1)
print('='*150)
print(result2)

{'리뷰내용': '배송이 정말 빠르고 제품이 만족스러워요', 'output': {'sentiment': 'positive', 'keyword': '배송, 품질, 만족', 'improvement': '없음'}}
{'리뷰내용': '배송이 정말 빠르고 제품이 만족스러워요', 'sentiment': 'positive', 'keyword': '배송, 품질, 만족', 'improvement': '없음'}
